In [1]:
from __future__ import annotations

import logging
from dataclasses import dataclass
from typing import Any, Optional

from datasets import Audio, DatasetDict, load_dataset
from omegaconf import DictConfig, OmegaConf

logger = logging.getLogger(__name__)


SPOKEN_SWAG_REPO = "slprl/SpokenSwag"


def load_spoken_swag(
    cfg: DictConfig,
    *,
    sampling_rate: int = 24000,
) -> DatasetDict:
    """Load the SpokenSwag DPO dataset, ready for Moshi.

    Args:
        cfg: a Hydra config (see `config/data/spoken_swag.yaml`). Recognised
            keys:

            - `repo_id`: HF repo id (default `slprl/SpokenSwag`).
            - `cache_dir`: optional HF cache override.
            - `num_proc`: parallelism for `.filter` / `.cast_column`.
            - `repetition_filter` (bool): drop rows where `auto_bleu2`
              exceeds `max_auto_bleu`.
            - `max_auto_bleu` (float): threshold for the above.
            - `train_take` / `val_take` (int | None): optional truncation,
              handy for smoke tests.
        sampling_rate: target sample rate for the Audio columns. Moshi /
            Mimi expects 24 kHz, which is also SpokenSwag's native rate.

    Returns:
        A `DatasetDict` with `train` and `validation` splits.
    """
    repo_id = cfg.get("repo_id", SPOKEN_SWAG_REPO)
    cache_dir = cfg.get("cache_dir", None)
    num_proc = cfg.get("num_proc", 4)

    logger.info("Loading %s (cache_dir=%s)", repo_id, cache_dir)
    ds: DatasetDict = load_dataset(repo_id, cache_dir=cache_dir)

    # Re-sample audio columns to Moshi's expected rate.
    audio_feat = Audio(sampling_rate=sampling_rate)
    for col in ("prompt", "chosen", "rejected"):
        ds = ds.cast_column(col, audio_feat)

    if cfg.get("repetition_filter", False):
        max_auto_bleu = float(cfg.get("max_auto_bleu", 0.3))

        def keep(x):
            return x["auto_bleu2"] is None or x["auto_bleu2"] < max_auto_bleu

        before = {k: len(v) for k, v in ds.items()}
        ds = ds.filter(keep, num_proc=num_proc)
        after = {k: len(v) for k, v in ds.items()}
        logger.info("repetition_filter %s -> %s (max_auto_bleu=%s)",
                    before, after, max_auto_bleu)

    train_take = cfg.get("train_take", None)
    val_take = cfg.get("val_take", None)
    if train_take is not None and "train" in ds:
        ds["train"] = ds["train"].select(range(min(train_take, len(ds["train"]))))
    if val_take is not None and "validation" in ds:
        ds["validation"] = ds["validation"].select(
            range(min(val_take, len(ds["validation"])))
        )

    return ds

In [2]:
conf = OmegaConf.load('spoken_swag.yaml')

In [ ]:
ds = load_spoken_swag(conf, sampling_rate=24000)

Resolving data files:   0%|          | 0/58 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/58 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

data/train-00000-of-00058.parquet:   0%|          | 0.00/434M [00:00<?, ?B/s]

...

data/train-00018-of-00058.parquet:   0%|          | 0.00/440M [00:00<?, ?B/s]

data/train-00019-of-00058.parquet:   0%|          | 0.00/440M [00:00<?, ?B/s]

data/train-00020-of-00058.parquet:   0%|          | 0.00/441M [00:00<?, ?B/s]

data/train-00021-of-00058.parquet:   0%|          | 0.00/441M [00:00<?, ?B/s]

data/train-00022-of-00058.parquet:   0%|          | 0.00/438M [00:00<?, ?B/s]